In [22]:
!pip install pandas numpy scikit-learn nltk Sastrawi gensim gradio

# Data Collection & Text Preprocessing

Import Library

In [23]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

Data Collection

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
path = "/content/drive/MyDrive/Natural Language Processing (NLP) /UTS"

In [26]:
df = pd.read_csv(path+'/tokopedia-product-reviews-2019.csv')
df = df.dropna(subset=['text'])

Buat Label Sentimen (0 = Negatif, 1 = Positif)

In [27]:
df['label'] = df['rating'].apply(lambda x: 1 if x > 3 else 0)

In [28]:
df_positif = df[df['label'] == 1]
df_negatif = df[df['label'] == 0]

In [29]:
min_sample = min(len(df_positif), len(df_negatif), 5000)
df_balanced = pd.concat([df_positif.sample(min_sample, random_state=42),
                         df_negatif.sample(min_sample, random_state=42)])

In [30]:
df = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

In [31]:
df.head()

,Unnamed: 0,text,rating,category,product_name,product_id,sold,shop_id,product_url,label
0,2350,Baguuusssss barangnyaaaaaa,3,fashion,X04 Rak sepatu 4 SUSUN payung holder lemari sepatu murah,411651606,"2,9rb",258554,https://www.tokopedia.com/nipponpowerbank/x04-rak-sepatu-4-susun-payung-holder-lemari-sepatu-murah,0
1,32227,Trimakasih. Pengiriman Cepat!,5,elektronik,CATRIDGE CANON PG 830 BLACK ORIGINAL 100%,158643386,36,1884654,https://www.tokopedia.com/sjkomputer/catridge-canon-pg-830-black-original-100,1
2,8447,Kualitas sesuai harga lah...barang ringkih banget,3,handphone,&#40;FJ010&#41; HEADPHONE JBL SOLO HD / HEADSHET JBL MODEL SOLO HD + MIC,269818535,125,113813,https://www.tokopedia.com/handphonetiam/fj010-headphone-jbl-solo-hd-headshet-jbl-model-solo-hd-mic,0
3,26219,"Barang yang dikirim berbeda dengan deskripsi dan gambar,\nbarang yang dikirim Sodim RAM 8GB DDR4 2Rx8 PC4-2400P-SE0-10 seharusnya Sodim RAM DDR4 4GB PC4-2400T-SC0-11",3,elektronik,Sodim RAM DDR4 4GB PC4-2400T Ram laptop sodimm 4GB DDR4-2400 PC4-2400,333814921,31,1787060,https://www.tokopedia.com/askompute/sodim-ram-ddr4-4gb-pc4-2400t-ram-laptop-sodimm-4gb-ddr4-2400-pc4-2400,0
4,5337,"Produk sesuai harga, kemasan packingannya bagus.",4,handphone,NOKIA 130 dual sim handphone hp,266910555,"2,7rb",2270419,https://www.tokopedia.com/anshopp123/nokia-130-dual-sim-handphone-hp,1


TEXT PREPROCESSING

In [32]:
factory_stop = StopWordRemoverFactory()
stopwords_id = factory_stop.get_stop_words()

factory_stem = StemmerFactory()
stemmer = factory_stem.create_stemmer()

nltk.download('punkt_tab')

def preprocess_text(text):
    # Case Folding
    text = text.lower()
    # Hapus URL, Mention, Hashtag, Angka, dan Tanda Baca
    text = re.sub(r'http\S+|www\S+|@\w+|#\w+|[^a-z\s]', '', text)
    # Mengubah huruf yang diketik berulang (lebih dari 1) menjadi satu huruf saja
    text = re.sub(r'([a-z])\1+', r'\1', text)
    # Tokenization
    tokens = word_tokenize(text)
    # Stopword Removal
    tokens = [word for word in tokens if word not in stopwords_id]
    # Stemming
    tokens = [stemmer.stem(word) for word in tokens]
    return tokens # Mengembalikan list token untuk Word2Vec

# Terapkan fungsi
print("Sedang memproses teks...")
df['tokens'] = df['text'].apply(preprocess_text)
df['cleaned_text'] = df['tokens'].apply(lambda x: ' '.join(x))
print("Preprocessing selesai!")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Sedang memproses teks...
Preprocessing selesai!


In [33]:
# Menampilkan perbandingan teks sebelum dan sesudah preprocessing
print("=== PERBANDINGAN TEKS ASLI VS TEKS BERSIH ===")
# atur max_colwidth agar teks yang panjang tidak terpotong
pd.set_option('display.max_colwidth', None)

# Tampilkan 10 baris pertama khusus kolom 'text' dan 'cleaned_text'
display(df[['text', 'cleaned_text']].head(10))

=== PERBANDINGAN TEKS ASLI VS TEKS BERSIH ===


,text,cleaned_text
0,Baguuusssss barangnyaaaaaa,bagus barang
1,Trimakasih. Pengiriman Cepat!,trimakasih kirim cepat
2,Kualitas sesuai harga lah...barang ringkih banget,kualitas sesuai harga lahbarang ringkih banget
3,"Barang yang dikirim berbeda dengan deskripsi dan gambar,\nbarang yang dikirim Sodim RAM 8GB DDR4 2Rx8 PC4-2400P-SE0-10 seharusnya Sodim RAM DDR4 4GB PC4-2400T-SC0-11",barang kirim beda deskripsi gambar barang kirim sodim ram gb dr rx pcpse sodim ram dr gb pctsc
4,"Produk sesuai harga, kemasan packingannya bagus.",produk sesuai harga kemas packinganya bagus
5,"Banyak yang ga lengket...., produk kurang bagus.",banyak ga lengket produk kurang bagus
6,"Barang sudah diterima, sesuai pesanan, makasih..",barang terima sesuai pesan makasih
7,"barang sudah diterima, mantap, dan biar bintang yang berbicara gan.................",barang terima mantap biar bintang bicara gan
8,Sesuai dengan yang diweb dan harga,sesuai diweb harga
9,Ukuran 42 nya ternyata kecil ngga sesuai deskripsi,ukur nya nyata kecil nga sesuai deskripsi


# Feature Extraction (TF-IDF & Word2Vec)

Import Library

In [34]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

TF-IDF

In [35]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf_vectorizer.fit_transform(df['cleaned_text'])

WORD2VEC

In [36]:
w2v_model = Word2Vec(sentences=df['tokens'], vector_size=100, window=5, min_count=2, workers=4)

In [37]:
def get_avg_word2vec(tokens, model, vector_size):
    valid_words = [word for word in tokens if word in model.wv.key_to_index]
    if not valid_words:
        return np.zeros(vector_size)
    return np.mean([model.wv[word] for word in valid_words], axis=0)

X_w2v = np.array([get_avg_word2vec(tokens, w2v_model, 100) for tokens in df['tokens']])
y = df['label'].values

# Modeling & Comparison

Import Library

In [38]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

Split Data

In [39]:
X_train_tf, X_test_tf, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)
X_train_w2v, X_test_w2v, _, _ = train_test_split(X_w2v, y, test_size=0.2, random_state=42)

Inisialisasi Model

In [40]:
models_tfidf = {
    "Naive Bayes (TF-IDF)": MultinomialNB(),
    "Logistic Regression (TF-IDF)": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "SVM (TF-IDF)": SVC(class_weight='balanced')
}

models_w2v = {
    "Naive Bayes (Word2Vec)": GaussianNB(),
    "Logistic Regression (Word2Vec)": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "SVM (Word2Vec)": SVC(class_weight='balanced')
}

def train_and_evaluate(models, X_train, X_test, y_train, y_test):
    results = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        print(f"--- {name} ---")
        print(classification_report(y_test, y_pred))
        results[name] = model
    return results

print("\nEVALUASI MODEL DENGAN TF-IDF")
trained_tfidf_models = train_and_evaluate(models_tfidf, X_train_tf, X_test_tf, y_train, y_test)

print("\nEVALUASI MODEL DENGAN WORD2VEC")
trained_w2v_models = train_and_evaluate(models_w2v, X_train_w2v, X_test_w2v, y_train, y_test)


EVALUASI MODEL DENGAN TF-IDF
--- Naive Bayes (TF-IDF) ---
              precision    recall  f1-score   support

           0       0.81      0.68      0.74       553
           1       0.72      0.84      0.78       547

    accuracy                           0.76      1100
   macro avg       0.77      0.76      0.76      1100
weighted avg       0.77      0.76      0.76      1100

--- Logistic Regression (TF-IDF) ---
              precision    recall  f1-score   support

           0       0.78      0.80      0.79       553
           1       0.79      0.77      0.78       547

    accuracy                           0.78      1100
   macro avg       0.78      0.78      0.78      1100
weighted avg       0.78      0.78      0.78      1100

--- SVM (TF-IDF) ---
              precision    recall  f1-score   support

           0       0.79      0.80      0.79       553
           1       0.79      0.78      0.79       547

    accuracy                           0.79      1100
   macro av

Simple Deployment dengan Gradio

In [41]:
import gradio as gr

# Pilih model untuk deployment, misalnya Logistic Regression dengan TF-IDF
best_model = trained_tfidf_models["Logistic Regression (TF-IDF)"]

def predict_sentiment(review_text):
    # Preprocess teks input pengguna
    tokens = preprocess_text(review_text)
    cleaned_str = ' '.join(tokens)

    # Ekstraksi Fitur
    vectorized_text = tfidf_vectorizer.transform([cleaned_str])

    # Prediksi
    prediction = best_model.predict(vectorized_text)[0]

    if prediction == 1:
        return "Positif 🟢"
    else:
        return "Negatif 🔴"

# Buat UI Gradio
interface = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(lines=3, placeholder="Masukkan ulasan produk Tokopedia di sini..."),
    outputs="text",
    title="Prediksi Sentimen Ulasan E-Commerce",
    description="Model Machine Learning untuk mendeteksi sentimen positif atau negatif dari ulasan produk."
)

# Jalankan Gradio
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://096fc945271e3c22d9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [42]:
import joblib

# Simpan Model Terbaik dan Vectorizer-nya
joblib.dump(trained_tfidf_models["Logistic Regression (TF-IDF)"], 'best_model_tfidf.pkl')
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')

print("Model dan Vectorizer berhasil disimpan!")

Model dan Vectorizer berhasil disimpan!
